In [1]:
# Import Packages
import pandas as pd
import re

# Import from local files
from data_cleaning import DataCleaner
from extraction import extract_data
from entity_resolution import EntityResolver

resolver = EntityResolver()

# Set pandas display options
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)


# ----------------------------------------------------------
# Extract
# ----------------------------------------------------------

extracted_df = extract_data(
    "2025 Roll - Parcels Export.xlsx",
    "Only Delinquent Parcels"
)


# ----------------------------------------------------------
# Clean
# ----------------------------------------------------------

cleaner = DataCleaner()

clean_df = cleaner.clean_data(
    extracted_df
)


# ----------------------------------------------------------
# Entity Resolution
# ----------------------------------------------------------

(
    entity_occurrences,
    entities,
    parcel_entity_roles,
    association_evidence,
    entity_associations
) = EntityResolver().build(
    clean_df
)

# Create audit dataframe.
resolution_audit = (
    entity_occurrences
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address",
                "entity_type",
                "occurrence_count"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one",
        suffixes=("_source", "_final")
    )
)

resolution_audit.head()

,occurrence_id,parcel_number,entity_name,associated_address,role,source_entity_name,source_column,source_occurrence_id,entity_type_source,joint_parse_rule,joint_parse_confidence,joint_expanded,joint_party_index,joint_party_count,organization_type,organization_classification_confidence,address_source,entity_id,canonical_name,canonical_address,entity_type_final,occurrence_count
0,1,01-01-201-013,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Owner,POLOMCAK CLAUDIA,owner_name,1,Person,,,False,NaN,NaN,NaN,NaN,owner_address,3587,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Person,2
1,2,01-01-251-013,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA &,owner_name,2,Joint,,,False,NaN,NaN,NaN,NaN,owner_address,1009,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Joint,2
2,3,01-01-251-050,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,3,Person,,,False,NaN,NaN,NaN,NaN,owner_address,1010,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
3,4,01-01-251-060,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,4,Person,,,False,NaN,NaN,NaN,NaN,owner_address,1010,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
4,5,01-01-301-050,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Owner,SPIEKERMANN TIM & JUDY,owner_name,5,Person,shared_surname_single_right_name,High,True,1.0,2.0,NaN,NaN,owner_address,4261,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Person,2


In [2]:
entities["occurrence_count"].sum()

np.int64(12058)

## Repeated surname audit

In [3]:
def has_repeated_surname_comma_structure(name):
    """
    Flag names that appear to contain two full person names
    separated by a comma, where both sides begin with the
    same surname.

    Example
    -------
    DARDEN MURPHY JR, DARDEN VICENT B
        -> True

    SMITH, JOHN A
        -> False
    """

    if pd.isna(name):
        return False

    name = str(name).upper().strip()

    # This audit is specifically for a two-part comma structure.
    parts = [
        part.strip()
        for part in name.split(",")
        if part.strip()
    ]

    if len(parts) != 2:
        return False

    left_tokens = re.findall(
        r"[A-Z]+(?:['-][A-Z]+)*",
        parts[0]
    )

    right_tokens = re.findall(
        r"[A-Z]+(?:['-][A-Z]+)*",
        parts[1]
    )

    # Each side should look substantial enough to represent
    # a surname plus at least one given-name token.
    if (
            len(left_tokens) < 2
            or len(right_tokens) < 2
    ):
        return False

    return left_tokens[0] == right_tokens[0]

In [4]:
repeating_surname_comma_audit = (
    entity_occurrences[
        (
            entity_occurrences["entity_type"]
            == "Person"
        )
        &
        (
            entity_occurrences["entity_name"]
            .apply(
                has_repeated_surname_comma_structure
            )
        )
    ]
    [
        [
            "entity_id",
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "parcel_number",
            "role",
            "associated_address",
            "source_column"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "entity_name",
            "parcel_number",
            "role"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Person occurrences with a repeated-surname "
    "comma structure:",
    len(repeating_surname_comma_audit)
)

print(
    "Distinct entity names:",
    repeating_surname_comma_audit[
        "entity_name"
    ].nunique()
)

repeating_surname_comma_audit

Person occurrences with a repeated-surname comma structure: 0
Distinct entity names: 0


,entity_id,source_occurrence_id,source_entity_name,entity_name,parcel_number,role,associated_address,source_column


In [5]:
repeating_surname_comma_summary = (
    repeating_surname_comma_audit
    .groupby(
        [
            "entity_id",
            "entity_name",
            "associated_address"
        ],
        dropna=False
    )
    .agg(
        occurrence_count=(
            "source_occurrence_id",
            "nunique"
        ),
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        ),
        roles=(
            "role",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        ),
        source_names=(
            "source_entity_name",
            lambda x:
                " | ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
    .sort_values(
        "entity_name"
    )
    .reset_index(drop=True)
)

print(
    "Distinct suspicious entities:",
    len(repeating_surname_comma_summary)
)

repeating_surname_comma_summary

Distinct suspicious entities: 0


,entity_id,entity_name,associated_address,occurrence_count,parcel_count,parcels,roles,source_names


In [6]:
darden_joint_check = (
    entity_occurrences[
        entity_occurrences[
            "source_entity_name"
        ].isin(
            [
                "DARDEN MURPHY JR, DARDEN VICENT B",
                "DARDEN RICKIE L, DARDEN VANASSA A"
            ]
        )
    ]
    [
        [
            "source_entity_name",
            "entity_name",
            "entity_type",
            "parcel_number",
            "role",
            "associated_address"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "source_entity_name",
            "role"
        ]
    )
    .reset_index(drop=True)
)

darden_joint_check

,source_entity_name,entity_name,entity_type,parcel_number,role,associated_address
0,"DARDEN MURPHY JR, DARDEN VICENT B","DARDEN MURPHY JR, DARDEN VICENT B",Joint,06-09-477-002,Owner,"628 LULU ST, KALAMAZOO, MI 49007"
1,"DARDEN RICKIE L, DARDEN VANASSA A","DARDEN RICKIE L, DARDEN VANASSA A",Joint,06-09-477-002,Owner Care Of,"628 LULU ST, KALAMAZOO, MI 49007"
2,"DARDEN RICKIE L, DARDEN VANASSA A","DARDEN RICKIE L, DARDEN VANASSA A",Joint,06-09-477-002,Taxpayer Care Of,"628 LULU ST, KALAMAZOO, MI 49007"


In [7]:
print(
    "Occurrence count:",
    entities["occurrence_count"].sum()
)

Occurrence count: 12058


In [8]:
repeating_surname_comma_audit = (
    entity_occurrences[
        (
            entity_occurrences["entity_type"]
            == "Person"
        )
        &
        (
            entity_occurrences["entity_name"]
            .apply(
                has_repeated_surname_comma_structure
            )
        )
    ]
)

print(
    "Remaining repeated-surname comma Persons:",
    len(repeating_surname_comma_audit)
)

Remaining repeated-surname comma Persons: 0


## Association evidence audit

In [9]:
print(
    "Association evidence rows:",
    len(association_evidence)
)

association_evidence[
    "evidence_strength"
].value_counts()

Association evidence rows: 508


evidence_strength
Medium    267
High      197
Low        44
Name: count, dtype: int64

In [10]:
association_evidence[
    "evidence_rule"
].value_counts()

evidence_rule
SHARED_OWNER_ADDRESS                          97
SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER         81
SHARED_TAXPAYER_ADDRESS                       69
SAME_PARCEL_CO_OWNER                          61
SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER         55
SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO      51
SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER      50
SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF    44
Name: count, dtype: int64

In [11]:
print(
    "Duplicate evidence rows:",
    association_evidence.duplicated(
        subset=[
            "entity_id_a",
            "entity_id_b",
            "evidence_rule",
            "parcel_number_a",
            "parcel_number_b",
            "role_a",
            "role_b"
        ]
    ).sum()
)

Duplicate evidence rows: 0


In [12]:
print(
    "Total association evidence rows:",
    len(association_evidence)
)

print("\nEvidence rules:")
print(
    association_evidence[
        "evidence_rule"
    ].value_counts()
)

print("\nEvidence strengths:")
print(
    association_evidence[
        "evidence_strength"
    ].value_counts()
)

Total association evidence rows: 508

Evidence rules:
evidence_rule
SHARED_OWNER_ADDRESS                          97
SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER         81
SHARED_TAXPAYER_ADDRESS                       69
SAME_PARCEL_CO_OWNER                          61
SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER         55
SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO      51
SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER      50
SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF    44
Name: count, dtype: int64

Evidence strengths:
evidence_strength
Medium    267
High      197
Low        44
Name: count, dtype: int64


In [13]:
shared_owner_evidence = (
    association_evidence[
        association_evidence[
            "evidence_rule"
        ]
        == "SHARED_OWNER_ADDRESS"
    ]
)

print(
    "Shared-owner-address evidence rows:",
    len(shared_owner_evidence)
)

print(
    "Distinct entity pairs:",
    shared_owner_evidence[
        [
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared addresses:",
    shared_owner_evidence[
        "shared_address"
    ].nunique()
)

print(
    "Duplicate shared-owner evidence:",
    shared_owner_evidence.duplicated(
        subset=[
            "entity_id_a",
            "entity_id_b",
            "evidence_rule",
            "shared_address"
        ]
    ).sum()
)

Shared-owner-address evidence rows: 97
Distinct entity pairs: 97
Distinct shared addresses: 52
Duplicate shared-owner evidence: 0


In [14]:
print(
    "Shared-owner rows with parcel_number_a populated:",
    shared_owner_evidence[
        "parcel_number_a"
    ].notna().sum()
)

print(
    "Shared-owner rows with parcel_number_b populated:",
    shared_owner_evidence[
        "parcel_number_b"
    ].notna().sum()
)

Shared-owner rows with parcel_number_a populated: 0
Shared-owner rows with parcel_number_b populated: 0


In [15]:
# Every evidence ID should be unique.
assert association_evidence[
    "evidence_id"
].is_unique

# Entity pairs should always be normalized.
assert (
    association_evidence[
        "entity_id_a"
    ]
    <
    association_evidence[
        "entity_id_b"
    ]
).all()

# Every referenced entity should actually exist.
valid_entity_ids = set(
    entities["entity_id"]
)

assert set(
    association_evidence["entity_id_a"]
).issubset(valid_entity_ids)

assert set(
    association_evidence["entity_id_b"]
).issubset(valid_entity_ids)

# Shared-owner-address evidence should have exactly
# the expected characteristics.
shared_owner = association_evidence[
    association_evidence["evidence_rule"]
    == "SHARED_OWNER_ADDRESS"
]

assert (
    shared_owner["evidence_strength"]
    == "Medium"
).all()

assert (
    shared_owner["address_type"]
    == "Owner"
).all()

assert (
    shared_owner["role_a"]
    == "Owner"
).all()

assert (
    shared_owner["role_b"]
    == "Owner"
).all()

assert shared_owner[
    "shared_address"
].notna().all()

assert (
    shared_owner[
        "shared_address"
    ].str.strip()
    != ""
).all()

print("Stage 5 shared-owner-address validation passed.")

Stage 5 shared-owner-address validation passed.


## Shared taxpayer address audit

In [16]:
# ----------------------------------------------------------
# One taxpayer address per parcel
# ----------------------------------------------------------

parcel_taxpayer_addresses = (
    clean_df[
        [
            "parcel_number",
            "taxpayer_address"
        ]
    ]
    .copy()
)

parcel_taxpayer_addresses = (
    parcel_taxpayer_addresses[
        (
            parcel_taxpayer_addresses[
                "taxpayer_address"
            ].notna()
        )
        &
        (
            parcel_taxpayer_addresses[
                "taxpayer_address"
            ].str.strip()
            != ""
        )
    ]
    .drop_duplicates()
)

In [17]:
taxpayer_address_counts = (
    parcel_taxpayer_addresses
    .groupby("parcel_number")[
        "taxpayer_address"
    ]
    .nunique()
)

print(
    "Parcels with more than one taxpayer address:",
    (taxpayer_address_counts > 1).sum()
)

Parcels with more than one taxpayer address: 0


In [18]:
owner_entities_by_parcel = (
    parcel_entity_roles[
        parcel_entity_roles[
            "role"
        ] == "Owner"
    ]
    [
        [
            "parcel_number",
            "entity_id"
        ]
    ]
    .drop_duplicates()
)

owner_taxpayer_address_entities = (
    owner_entities_by_parcel
    .merge(
        parcel_taxpayer_addresses,
        on="parcel_number",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        entities[
            [
                "entity_id",
                "entity_type",
                "canonical_name"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one"
    )
)

In [19]:
print(
    "Owner rows with missing entity type:",
    owner_taxpayer_address_entities[
        "entity_type"
    ].isna().sum()
)

Owner rows with missing entity type: 0


### Profile taxpayer addresses

In [20]:
taxpayer_address_profile = (
    owner_taxpayer_address_entities
    .groupby(
        "taxpayer_address"
    )
    .agg(
        distinct_entities=(
            "entity_id",
            "nunique"
        ),
        distinct_parcels=(
            "parcel_number",
            "nunique"
        )
    )
    .reset_index()
)

In [21]:
person_counts = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Person"
    ]
    .groupby(
        "taxpayer_address"
    )[
        "entity_id"
    ]
    .nunique()
    .rename(
        "distinct_person_entities"
    )
)

organization_counts = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    .groupby(
        "taxpayer_address"
    )[
        "entity_id"
    ]
    .nunique()
    .rename(
        "distinct_organization_entities"
    )
)

taxpayer_address_profile = (
    taxpayer_address_profile
    .merge(
        person_counts,
        on="taxpayer_address",
        how="left"
    )
    .merge(
        organization_counts,
        on="taxpayer_address",
        how="left"
    )
    .fillna(
        {
            "distinct_person_entities": 0,
            "distinct_organization_entities": 0
        }
    )
)

taxpayer_address_profile[
    [
        "distinct_person_entities",
        "distinct_organization_entities"
    ]
] = (
    taxpayer_address_profile[
        [
            "distinct_person_entities",
            "distinct_organization_entities"
        ]
    ]
    .astype(int)
)

taxpayer_address_profile = (
    taxpayer_address_profile
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

In [22]:
print(
    "Taxpayer addresses:",
    len(taxpayer_address_profile)
)

print(
    "Addresses linked to >1 owner entity:",
    (
        taxpayer_address_profile[
            "distinct_entities"
        ] > 1
    ).sum()
)

print(
    "Addresses containing both Person and Organization owners:",
    (
        (
            taxpayer_address_profile[
                "distinct_person_entities"
            ] > 0
        )
        &
        (
            taxpayer_address_profile[
                "distinct_organization_entities"
            ] > 0
        )
    ).sum()
)

taxpayer_address_profile[
    [
        "distinct_entities",
        "distinct_parcels"
    ]
].describe()

Taxpayer addresses: 1558
Addresses linked to >1 owner entity: 440
Addresses containing both Person and Organization owners: 54


,distinct_entities,distinct_parcels
count,1558.000000,1558.000000
mean,1.352375,1.269576
std,0.712106,1.058585
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,1.000000
max,14.000000,20.000000


In [23]:
taxpayer_address_persons = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Person"
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "canonical_name",
            "taxpayer_address"
        ]
    ]
    .rename(
        columns={
            "parcel_number":
                "person_parcel",
            "entity_id":
                "person_id",
            "canonical_name":
                "person_name"
        }
    )
)

taxpayer_address_organizations = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "canonical_name",
            "taxpayer_address"
        ]
    ]
    .rename(
        columns={
            "parcel_number":
                "organization_parcel",
            "entity_id":
                "organization_id",
            "canonical_name":
                "organization_name"
        }
    )
)

shared_taxpayer_address_candidates = (
    taxpayer_address_organizations
    .merge(
        taxpayer_address_persons,
        on="taxpayer_address",
        how="inner"
    )
)

shared_taxpayer_address_candidates = (
    shared_taxpayer_address_candidates[
        shared_taxpayer_address_candidates[
            "organization_parcel"
        ]
        !=
        shared_taxpayer_address_candidates[
            "person_parcel"
        ]
    ]
    .copy()
)

In [24]:
print(
    "Cross-parcel shared-taxpayer-address candidate rows:",
    len(shared_taxpayer_address_candidates)
)

print(
    "Distinct Person-Organization pairs:",
    shared_taxpayer_address_candidates[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared taxpayer addresses:",
    shared_taxpayer_address_candidates[
        "taxpayer_address"
    ].nunique()
)

print(
    "Distinct Person-Organization-address combinations:",
    shared_taxpayer_address_candidates[
        [
            "person_id",
            "organization_id",
            "taxpayer_address"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Cross-parcel shared-taxpayer-address candidate rows: 150
Distinct Person-Organization pairs: 69
Distinct shared taxpayer addresses: 36
Distinct Person-Organization-address combinations: 69


In [25]:
candidate_taxpayer_addresses = set(
    shared_taxpayer_address_candidates[
        "taxpayer_address"
    ]
)

taxpayer_address_member_summary = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "taxpayer_address"
        ].isin(
            candidate_taxpayer_addresses
        )
    ]
    .groupby(
        [
            "taxpayer_address",
            "entity_id",
            "canonical_name",
            "entity_type"
        ],
        dropna=False
    )
    .agg(
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna()
                            .astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
    .merge(
        taxpayer_address_profile[
            [
                "taxpayer_address",
                "distinct_entities",
                "distinct_parcels",
                "distinct_person_entities",
                "distinct_organization_entities"
            ]
        ],
        on="taxpayer_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "taxpayer_address",
            "entity_type",
            "canonical_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

taxpayer_address_member_summary

,taxpayer_address,entity_id,canonical_name,entity_type,parcel_count,parcels,distinct_entities,distinct_parcels,distinct_person_entities,distinct_organization_entities
0,"218 GLADYS ST, PORTAGE, MI 49002",3031,MIKA FIRST LLC,Organization,3,"06-10-315-021, 06-10-389-037, 06-16-248-002",5,7,2,3
1,"218 GLADYS ST, PORTAGE, MI 49002",4214,SMS HOLDINGS LLC,Organization,1,06-10-377-011,5,7,2,3
2,"218 GLADYS ST, PORTAGE, MI 49002",4215,SMS MICHIGAN HOLDINGS LLC,Organization,1,06-09-397-007,5,7,2,3
3,"218 GLADYS ST, PORTAGE, MI 49002",630,"BRINKS, TERRY",Person,1,06-16-328-005,5,7,2,3
4,"218 GLADYS ST, PORTAGE, MI 49002",4476,"THOMAS, TIMOTHY",Person,1,06-02-320-010,5,7,2,3
...,...,...,...,...,...,...,...,...,...,...
100,"3131 BRYNMAWR DR, PORTAGE, MI 49024-5654",662,"BROOKS, RONALD",Person,1,06-10-398-036,2,2,1,1
101,"4205 AMELIA WAY, NAPLES, FL 34119-9043",4145,"SKARTSIARIS, THEODORE MARITAL TRST",Organization,1,06-34-365-001,2,2,1,1
102,"4205 AMELIA WAY, NAPLES, FL 34119-9043",4144,"SKARTSIARIS, THEODORE E, TRUSTEE",Person,1,06-15-354-130,2,2,1,1
103,"51842 WELCHER DR, MARCELLUS, MI 49067",1769,"GOVE, DOUGLAS ESTATE",Organization,1,06-15-127-001,2,2,1,1


In [26]:
taxpayer_address_member_summary.to_csv(
    "taxpayer_address_member_summary.csv",
    index=False
)

shared_taxpayer_address_candidates.to_csv(
    "shared_taxpayer_address_candidates.csv",
    index=False
)

In [27]:
print(
    "Parcels with more than one taxpayer address:",
    (taxpayer_address_counts > 1).sum()
)

print(
    "Owner rows with missing entity type:",
    owner_taxpayer_address_entities[
        "entity_type"
    ].isna().sum()
)

Parcels with more than one taxpayer address: 0
Owner rows with missing entity type: 0


In [28]:
shared_taxpayer_address_candidates

,organization_parcel,organization_id,organization_name,taxpayer_address,person_parcel,person_id,person_name
2,06-09-397-007,4215,SMS MICHIGAN HOLDINGS LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-02-320-010,4476,"THOMAS, TIMOTHY"
3,06-09-397-007,4215,SMS MICHIGAN HOLDINGS LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-16-328-005,630,"BRINKS, TERRY"
4,06-10-311-025,1364,"DREAM MANAGEMENT, LLC","1103 WOODWARD AVE, KALAMAZOO, MI 49007",06-10-380-023,181,"ANDERSON, DEJA"
5,06-10-315-021,3031,MIKA FIRST LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-02-320-010,4476,"THOMAS, TIMOTHY"
6,06-10-315-021,3031,MIKA FIRST LLC,"218 GLADYS ST, PORTAGE, MI 49002",06-16-328-005,630,"BRINKS, TERRY"
...,...,...,...,...,...,...,...
181,07-24-215-160,4453,THE COPPER CREEK TRUST,"4960 S 36TH ST, CLIMAX, MI 49034",07-24-215-100,1784,"GRAHAM, WILLIAM D"
182,07-24-215-160,4453,THE COPPER CREEK TRUST,"4960 S 36TH ST, CLIMAX, MI 49034",07-24-215-110,1784,"GRAHAM, WILLIAM D"
184,08-18-301-140,2803,MARION CHARLES & BETTY TRUST,"12195 E MICHIGAN, GALESBURG, MI 49053",06-23-363-453,1866,"HAIST, MARCIA MAY"
185,08-18-301-140,2803,MARION CHARLES & BETTY TRUST,"12195 E MICHIGAN, GALESBURG, MI 49053",07-26-255-030,2805,"MARION, BETTY"


### Final shared-taxpayer-address validations

In [29]:
shared_taxpayer_evidence = (
    association_evidence[
        association_evidence[
            "evidence_rule"
        ] == "SHARED_TAXPAYER_ADDRESS"
    ]
    .copy()
)

print(
    "Total association evidence rows:",
    len(association_evidence)
)

print(
    "\nEvidence rules:"
)

print(
    association_evidence[
        "evidence_rule"
    ].value_counts()
)

print(
    "\nEvidence strengths:"
)

print(
    association_evidence[
        "evidence_strength"
    ].value_counts()
)

print(
    "\nShared-taxpayer-address evidence rows:",
    len(shared_taxpayer_evidence)
)

print(
    "Distinct entity pairs:",
    shared_taxpayer_evidence[
        [
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared taxpayer addresses:",
    shared_taxpayer_evidence[
        "shared_address"
    ].nunique()
)

print(
    "Duplicate shared-taxpayer evidence:",
    shared_taxpayer_evidence.duplicated(
        subset=[
            "entity_id_a",
            "entity_id_b",
            "evidence_rule",
            "shared_address"
        ]
    ).sum()
)

Total association evidence rows: 508

Evidence rules:
evidence_rule
SHARED_OWNER_ADDRESS                          97
SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER         81
SHARED_TAXPAYER_ADDRESS                       69
SAME_PARCEL_CO_OWNER                          61
SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER         55
SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO      51
SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER      50
SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF    44
Name: count, dtype: int64

Evidence strengths:
evidence_strength
Medium    267
High      197
Low        44
Name: count, dtype: int64

Shared-taxpayer-address evidence rows: 69
Distinct entity pairs: 69
Distinct shared taxpayer addresses: 36
Duplicate shared-taxpayer evidence: 0


In [30]:
# ==========================================================
# STRUCTURAL ASSERTIONS
# ==========================================================

# Evidence IDs should be unique.
assert association_evidence[
    "evidence_id"
].is_unique


# Entity pairs should always be normalized.
assert (
    association_evidence[
        "entity_id_a"
    ]
    <
    association_evidence[
        "entity_id_b"
    ]
).all()


# Every entity referenced by evidence must exist.
valid_entity_ids = set(
    entities["entity_id"]
)

assert set(
    association_evidence[
        "entity_id_a"
    ]
).issubset(
    valid_entity_ids
)

assert set(
    association_evidence[
        "entity_id_b"
    ]
).issubset(
    valid_entity_ids
)


# Stage 5C should contain exactly 69 rows.
assert len(
    shared_taxpayer_evidence
) == 69


# Every Stage 5C record should be Medium.
assert (
    shared_taxpayer_evidence[
        "evidence_strength"
    ]
    == "Medium"
).all()


# Address type should correctly identify taxpayer address.
assert (
    shared_taxpayer_evidence[
        "address_type"
    ]
    == "Taxpayer"
).all()


# Both associated entities participate as parcel Owners.
assert (
    shared_taxpayer_evidence[
        "role_a"
    ]
    == "Owner"
).all()

assert (
    shared_taxpayer_evidence[
        "role_b"
    ]
    == "Owner"
).all()


# Shared address must always be populated.
assert shared_taxpayer_evidence[
    "shared_address"
].notna().all()

assert (
    shared_taxpayer_evidence[
        "shared_address"
    ]
    .str.strip()
    != ""
).all()


# Parcel fields should remain empty because no single parcel
# pair is the evidence event.
assert shared_taxpayer_evidence[
    "parcel_number_a"
].isna().all()

assert shared_taxpayer_evidence[
    "parcel_number_b"
].isna().all()


# No duplicate Person-Organization-address evidence.
assert not shared_taxpayer_evidence.duplicated(
    subset=[
        "entity_id_a",
        "entity_id_b",
        "shared_address"
    ]
).any()

In [31]:
# ==========================================================
# ENTITY-TYPE VALIDATION
# ==========================================================

entity_type_lookup = (
    entities[
        [
            "entity_id",
            "entity_type"
        ]
    ]
    .set_index(
        "entity_id"
    )[
        "entity_type"
    ]
    .to_dict()
)

stage5c_entity_types = (
    shared_taxpayer_evidence[
        [
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .copy()
)

stage5c_entity_types[
    "entity_type_a"
] = (
    stage5c_entity_types[
        "entity_id_a"
    ]
    .map(
        entity_type_lookup
    )
)

stage5c_entity_types[
    "entity_type_b"
] = (
    stage5c_entity_types[
        "entity_id_b"
    ]
    .map(
        entity_type_lookup
    )
)

valid_type_pairs = (
    stage5c_entity_types
    .apply(
        lambda row:
            {
                row["entity_type_a"],
                row["entity_type_b"]
            }
            ==
            {
                "Person",
                "Organization"
            },
        axis=1
    )
)

print(
    "Invalid Person-Organization type pairs:",
    (~valid_type_pairs).sum()
)

assert valid_type_pairs.all()

Invalid Person-Organization type pairs: 0


In [32]:
# ==========================================================
# AUDIT-TO-PRODUCTION RECONCILIATION
# ==========================================================

expected_stage5c = (
    shared_taxpayer_address_candidates[
        [
            "person_id",
            "organization_id",
            "taxpayer_address"
        ]
    ]
    .drop_duplicates()
    .copy()
)

expected_stage5c[
    "entity_id_a"
] = (
    expected_stage5c[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .min(
        axis=1
    )
)

expected_stage5c[
    "entity_id_b"
] = (
    expected_stage5c[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .max(
        axis=1
    )
)

expected_stage5c = (
    expected_stage5c[
        [
            "entity_id_a",
            "entity_id_b",
            "taxpayer_address"
        ]
    ]
    .rename(
        columns={
            "taxpayer_address":
                "shared_address"
        }
    )
    .drop_duplicates()
    .sort_values(
        [
            "entity_id_a",
            "entity_id_b",
            "shared_address"
        ]
    )
    .reset_index(drop=True)
)

actual_stage5c = (
    shared_taxpayer_evidence[
        [
            "entity_id_a",
            "entity_id_b",
            "shared_address"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "entity_id_a",
            "entity_id_b",
            "shared_address"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Expected Stage 5C rows:",
    len(expected_stage5c)
)

print(
    "Actual Stage 5C rows:",
    len(actual_stage5c)
)

print(
    "Exact audit/production match:",
    expected_stage5c.equals(
        actual_stage5c
    )
)

assert expected_stage5c.equals(
    actual_stage5c
)

Expected Stage 5C rows: 69
Actual Stage 5C rows: 69
Exact audit/production match: False


AssertionError: 

In [ ]:
print(
    "Stage 5C shared-taxpayer-address validation passed."
)

In [33]:
# ==========================================================
# STAGE 5C RECONCILIATION DIAGNOSTIC
# ==========================================================

print("Expected dtypes:")
print(expected_stage5c.dtypes)

print("\nActual dtypes:")
print(actual_stage5c.dtypes)

print("\nExpected shape:")
print(expected_stage5c.shape)

print("\nActual shape:")
print(actual_stage5c.shape)

Expected dtypes:
entity_id_a       int64
entity_id_b       int64
shared_address      str
dtype: object

Actual dtypes:
entity_id_a        int64
entity_id_b        int64
shared_address    object
dtype: object

Expected shape:
(69, 3)

Actual shape:
(69, 3)


In [34]:
expected_stage5c_check = (
    expected_stage5c.copy()
)

actual_stage5c_check = (
    actual_stage5c.copy()
)

for df in [
    expected_stage5c_check,
    actual_stage5c_check
]:
    df["entity_id_a"] = (
        df["entity_id_a"]
        .astype("int64")
    )

    df["entity_id_b"] = (
        df["entity_id_b"]
        .astype("int64")
    )

    df["shared_address"] = (
        df["shared_address"]
        .astype(str)
        .str.strip()
    )

    df.sort_values(
        [
            "entity_id_a",
            "entity_id_b",
            "shared_address"
        ],
        inplace=True,
        kind="stable"
    )

    df.reset_index(
        drop=True,
        inplace=True
    )

print(
    "\nExact match after dtype normalization:",
    expected_stage5c_check.equals(
        actual_stage5c_check
    )
)


Exact match after dtype normalization: True


In [35]:
expected_keys = set(
    expected_stage5c_check[
        [
            "entity_id_a",
            "entity_id_b",
            "shared_address"
        ]
    ]
    .itertuples(
        index=False,
        name=None
    )
)

actual_keys = set(
    actual_stage5c_check[
        [
            "entity_id_a",
            "entity_id_b",
            "shared_address"
        ]
    ]
    .itertuples(
        index=False,
        name=None
    )
)

print(
    "Expected evidence keys:",
    len(expected_keys)
)

print(
    "Actual evidence keys:",
    len(actual_keys)
)

print(
    "Exact evidence-key match:",
    expected_keys == actual_keys
)

assert expected_keys == actual_keys

print(
    "Stage 5C audit-to-production "
    "reconciliation passed."
)

Expected evidence keys: 69
Actual evidence keys: 69
Exact evidence-key match: True
Stage 5C audit-to-production reconciliation passed.


## Stage 6

## Owner address audit

In [ ]:
owner_address_entities = (
    entity_occurrences[
        (
            entity_occurrences["role"]
            == "Owner"
        )
        &
        (
            entity_occurrences[
                "associated_address"
            ].notna()
        )
        &
        (
            entity_occurrences[
                "associated_address"
            ].str.strip()
            != ""
        )
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "associated_address"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "associated_address":
                "owner_address"
        }
    )
)

In [ ]:
owner_address_entities = (
    owner_address_entities
    .merge(
        entities[
            [
                "entity_id",
                "entity_type",
                "canonical_name"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
owner_address_profile = (
    owner_address_entities
    .groupby(
        "owner_address"
    )
    .agg(
        distinct_entities=(
            "entity_id",
            "nunique"
        ),
        distinct_parcels=(
            "parcel_number",
            "nunique"
        ),
        distinct_person_entities=(
            "entity_id",
            lambda ids:
                owner_address_entities.loc[
                    ids.index
                ]
                .loc[
                    lambda df:
                        df["entity_type"]
                        == "Person",
                    "entity_id"
                ]
                .nunique()
        ),
        distinct_organization_entities=(
            "entity_id",
            lambda ids:
                owner_address_entities.loc[
                    ids.index
                ]
                .loc[
                    lambda df:
                        df["entity_type"]
                        == "Organization",
                    "entity_id"
                ]
                .nunique()
        )
    )
    .reset_index()
)

owner_address_profile.head(30)

In [ ]:
owner_address_profile[
    [
        "distinct_entities",
        "distinct_parcels"
    ]
].describe()

In [ ]:
owner_address_persons = (
    owner_address_entities[
        owner_address_entities[
            "entity_type"
        ] == "Person"
    ]
    .rename(
        columns={
            "entity_id":
                "person_id",
            "canonical_name":
                "person_name",
            "parcel_number":
                "person_parcel"
        }
    )
)

owner_address_organizations = (
    owner_address_entities[
        owner_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    .rename(
        columns={
            "entity_id":
                "organization_id",
            "canonical_name":
                "organization_name",
            "parcel_number":
                "organization_parcel"
        }
    )
)

shared_owner_address_candidates = (
    owner_address_organizations
    .merge(
        owner_address_persons,
        on="owner_address",
        how="inner"
    )
)

In [ ]:
shared_owner_address_candidates = (
    shared_owner_address_candidates[
        shared_owner_address_candidates[
            "organization_parcel"
        ]
        !=
        shared_owner_address_candidates[
            "person_parcel"
        ]
    ]
    .copy()
)

In [ ]:
print(
    "Cross-parcel shared-owner-address evidence rows:",
    len(shared_owner_address_candidates)
)

print(
    "Distinct Person-Organization pairs:",
    shared_owner_address_candidates[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared owner addresses:",
    shared_owner_address_candidates[
        "owner_address"
    ].nunique()
)

In [ ]:
candidate_address_degree = (
    shared_owner_address_candidates[
        [
            "owner_address"
        ]
    ]
    .drop_duplicates()
    .merge(
        owner_address_profile,
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels"
        ],
        ascending=False
    )
)

candidate_address_degree.head(30)

In [ ]:
owner_address_candidate_detail = (
    shared_owner_address_candidates[
        [
            "owner_address",
            "person_id",
            "person_name",
            "person_parcel",
            "organization_id",
            "organization_name",
            "organization_parcel"
        ]
    ]
    .merge(
        owner_address_profile[
            [
                "owner_address",
                "distinct_entities",
                "distinct_parcels"
            ]
        ],
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "owner_address",
            "organization_name",
            "person_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

owner_address_candidate_detail

In [ ]:
person_address_members = (
    owner_address_persons[
        [
            "owner_address",
            "person_id",
            "person_name",
            "person_parcel"
        ]
    ]
    .rename(
        columns={
            "person_id": "entity_id",
            "person_name": "entity_name",
            "person_parcel": "parcel_number"
        }
    )
    .assign(
        entity_type="Person"
    )
)

organization_address_members = (
    owner_address_organizations[
        [
            "owner_address",
            "organization_id",
            "organization_name",
            "organization_parcel"
        ]
    ]
    .rename(
        columns={
            "organization_id": "entity_id",
            "organization_name": "entity_name",
            "organization_parcel": "parcel_number"
        }
    )
    .assign(
        entity_type="Organization"
    )
)

candidate_addresses = set(
    shared_owner_address_candidates[
        "owner_address"
    ]
)

owner_address_members = (
    pd.concat(
        [
            person_address_members,
            organization_address_members
        ],
        ignore_index=True
    )
)

owner_address_members = (
    owner_address_members[
        owner_address_members[
            "owner_address"
        ].isin(candidate_addresses)
    ]
    .drop_duplicates()
)

In [ ]:
owner_address_member_summary = (
    owner_address_members
    .groupby(
        [
            "owner_address",
            "entity_id",
            "entity_name",
            "entity_type"
        ],
        dropna=False
    )
    .agg(
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(set(x))
                )
        )
    )
    .reset_index()
    .merge(
        owner_address_profile[
            [
                "owner_address",
                "distinct_entities",
                "distinct_parcels"
            ]
        ],
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "owner_address",
            "entity_type",
            "entity_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

owner_address_member_summary

## Audit of role combinations between different entities on the same parcel

In [ ]:
parcel_role_pairs = (
    parcel_entity_roles
    .merge(
        parcel_entity_roles,
        on="parcel_number",
        suffixes=("_a", "_b")
    )
)

# Remove self-pairs and keep each unordered entity pair once
parcel_role_pairs = (
    parcel_role_pairs[
        parcel_role_pairs["entity_id_a"]
        < parcel_role_pairs["entity_id_b"]
    ]
    .copy()
)

In [ ]:
entity_lookup_a = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id": "entity_id_a",
            "canonical_name": "name_a",
            "entity_type": "entity_type_a",
            "organization_type": "organization_type_a"
        }
    )
)

entity_lookup_b = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id": "entity_id_b",
            "canonical_name": "name_b",
            "entity_type": "entity_type_b",
            "organization_type": "organization_type_b"
        }
    )
)

parcel_role_pairs = (
    parcel_role_pairs
    .merge(
        entity_lookup_a,
        on="entity_id_a",
        how="left"
    )
    .merge(
        entity_lookup_b,
        on="entity_id_b",
        how="left"
    )
)

In [ ]:
parcel_role_pairs["role_pair"] = (
    parcel_role_pairs[
        ["role_a", "role_b"]
    ]
    .apply(
        lambda row: " + ".join(
            sorted(row)
        ),
        axis=1
    )
)

In [ ]:
parcel_role_pairs["entity_type_pair"] = (
    parcel_role_pairs[
        [
            "entity_type_a",
            "entity_type_b"
        ]
    ]
    .apply(
        lambda row: " + ".join(
            sorted(row)
        ),
        axis=1
    )
)

In [ ]:
role_pair_counts = (
    parcel_role_pairs[
        "role_pair"
    ]
    .value_counts()
    .rename_axis("role_pair")
    .reset_index(name="count")
)

role_pair_counts

In [ ]:
role_type_pair_counts = (
    parcel_role_pairs
    .groupby(
        [
            "role_pair",
            "entity_type_pair"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

role_type_pair_counts

In [ ]:
print(
    "Parcels containing more than one entity:",
    parcel_role_pairs[
        "parcel_number"
    ].nunique()
)

print(
    "Distinct same-parcel entity pairs:",
    parcel_role_pairs[
        [
            "parcel_number",
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
org_person_pairs = (
    parcel_role_pairs[
        (
            (
                parcel_role_pairs["entity_type_a"]
                == "Organization"
            )
            &
            (
                parcel_role_pairs["entity_type_b"]
                == "Person"
            )
        )
        |
        (
            (
                parcel_role_pairs["entity_type_a"]
                == "Person"
            )
            &
            (
                parcel_role_pairs["entity_type_b"]
                == "Organization"
            )
        )
    ]
    .copy()
)

In [ ]:
org_person_pairs["organization_id"] = (
    org_person_pairs.apply(
        lambda row:
            row["entity_id_a"]
            if row["entity_type_a"] == "Organization"
            else row["entity_id_b"],
        axis=1
    )
)

org_person_pairs["organization_name"] = (
    org_person_pairs.apply(
        lambda row:
            row["name_a"]
            if row["entity_type_a"] == "Organization"
            else row["name_b"],
        axis=1
    )
)

org_person_pairs["organization_role"] = (
    org_person_pairs.apply(
        lambda row:
            row["role_a"]
            if row["entity_type_a"] == "Organization"
            else row["role_b"],
        axis=1
    )
)

org_person_pairs["person_id"] = (
    org_person_pairs.apply(
        lambda row:
            row["entity_id_a"]
            if row["entity_type_a"] == "Person"
            else row["entity_id_b"],
        axis=1
    )
)

org_person_pairs["person_name"] = (
    org_person_pairs.apply(
        lambda row:
            row["name_a"]
            if row["entity_type_a"] == "Person"
            else row["name_b"],
        axis=1
    )
)

org_person_pairs["person_role"] = (
    org_person_pairs.apply(
        lambda row:
            row["role_a"]
            if row["entity_type_a"] == "Person"
            else row["role_b"],
        axis=1
    )
)

In [ ]:
org_person_role_counts = (
    org_person_pairs
    .groupby(
        [
            "organization_role",
            "person_role"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
    .reset_index(drop=True)
)

org_person_role_counts

In [ ]:
print(
    "Organization-Person evidence rows:",
    len(org_person_pairs)
)

print(
    "Distinct parcel/person/organization combinations:",
    org_person_pairs[
        [
            "parcel_number",
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct person/organization pairs across all parcels:",
    org_person_pairs[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
org_person_detail = (
    org_person_pairs[
        [
            "parcel_number",

            "person_id",
            "person_name",
            "person_role",

            "organization_id",
            "organization_name",
            "organization_type_a",
            "organization_type_b",
            "organization_role"
        ]
    ]
    .copy()
)

organization_lookup = (
    entities[
        [
            "entity_id",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "organization_id"
        }
    )
)

org_person_detail = (
    org_person_pairs[
        [
            "parcel_number",

            "person_id",
            "person_name",
            "person_role",

            "organization_id",
            "organization_name",
            "organization_role"
        ]
    ]
    .merge(
        organization_lookup,
        on="organization_id",
        how="left"
    )
    .sort_values(
        [
            "organization_role",
            "person_role",
            "parcel_number"
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
# Get the six most common directional role combinations
top_role_combinations = (
    org_person_role_counts
    .head(6)[
        [
            "organization_role",
            "person_role"
        ]
    ]
)

# Keep only rows belonging to those six combinations
top_role_candidates = (
    org_person_detail
    .merge(
        top_role_combinations,
        on=[
            "organization_role",
            "person_role"
        ],
        how="inner"
    )
)

# Randomize the order reproducibly, then take up to
# 10 examples from each role combination.
top_role_samples = (
    top_role_candidates
    .sample(
        frac=1,
        random_state=42
    )
    .groupby(
        [
            "organization_role",
            "person_role"
        ],
        group_keys=False,
        sort=False
    )
    .head(10)
    .sort_values(
        [
            "organization_role",
            "person_role",
            "parcel_number"
        ]
    )
    .reset_index(drop=True)
)

top_role_samples[
    [
        "parcel_number",
        "person_name",
        "person_role",
        "organization_name",
        "organization_type",
        "organization_role"
    ]
]

## Investigating person-like entities

In [ ]:
person_red_flag_pattern = (
    r"\b(?:"
    r"IRA|"
    r"FBO|"
    r"CUSTODIAN|"
    r"SOLE\s+PROPRIETORSHIP|"
    r"MGMT|"
    r"MANAGEMENT|"
    r"MORTGAGE|"
    r"BANK|"
    r"PROPERTIES|"
    r"HOLDINGS|"
    r"SERV|"
    r"SERVICES"
    r")\b"
)

person_classification_audit = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.contains(
                person_red_flag_pattern,
                regex=True,
                na=False
            )
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Potentially suspicious Person entities:",
    len(person_classification_audit)
)

person_classification_audit

In [ ]:
suspicious_names = [
    "FBO GARY E APPS IRA",
    "FBO JOHN MEYERS IRA",
    "FBO RAMON HUERTA IRA",
    "FBO RICHARD DAUDERT IRA",
    "RUSHMORE LOAD MGT SERV"
]

entities[
    entities["canonical_name"].isin(
        suspicious_names
    )
][
    [
        "entity_id",
        "canonical_name",
        "entity_type",
        "organization_type",
        "organization_classification_confidence",
        "canonical_address",
        "occurrence_count"
    ]
].sort_values(
    [
        "canonical_name",
        "canonical_address"
    ]
)

In [ ]:
entities["occurrence_count"].sum()

## Auditing multi-comma persons

In [ ]:
multi_comma_persons = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.count(",")
            >= 3
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Person entities with at least 3 commas:",
    len(multi_comma_persons)
)

multi_comma_persons

In [ ]:
two_comma_persons = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.count(",")
            == 2
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Person entities with exactly 2 commas:",
    len(two_comma_persons)
)

two_comma_persons

## Audit parcel_entity_roles

In [ ]:
print(parcel_entity_roles.head(10))

In [ ]:
print(
    "Entity occurrences:",
    len(entity_occurrences)
)

print(
    "Parcel entity roles:",
    len(parcel_entity_roles)
)

print(
    "Unique parcels:",
    parcel_entity_roles[
        "parcel_number"
    ].nunique()
)

print(
    "Unique entities represented:",
    parcel_entity_roles[
        "entity_id"
    ].nunique()
)

In [ ]:
duplicate_relationships = (
    parcel_entity_roles
    .duplicated(
        subset=[
            "parcel_number",
            "entity_id",
            "role"
        ]
    )
    .sum()
)

print(
    "Duplicate parcel/entity/role relationships:",
    duplicate_relationships
)

In [ ]:
expected_relationships = (
    entity_occurrences[
        [
            "parcel_number",
            "entity_id",
            "role"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Expected relationships:",
    len(expected_relationships)
)

print(
    "Actual relationships:",
    len(parcel_entity_roles)
)

assert (
    len(expected_relationships)
    == len(parcel_entity_roles)
)

In [ ]:
parcel_entity_roles[
    "role"
].value_counts()

In [ ]:
parcel_entity_roles.groupby(
    "parcel_number"
)["entity_id"].nunique().describe()

In [ ]:
role_occurrence_counts = (
    entity_occurrences
    .groupby(
        [
            "parcel_number",
            "entity_id",
            "role"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="occurrence_count"
    )
)

duplicate_role_relationships = (
    role_occurrence_counts[
        role_occurrence_counts[
            "occurrence_count"
        ] > 1
    ]
    .sort_values(
        by="occurrence_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Parcel/entity/role relationships represented "
    "by multiple occurrences:",
    len(duplicate_role_relationships)
)

duplicate_role_relationships

In [ ]:
duplicate_role_detail = (
    entity_occurrences
    .merge(
        duplicate_role_relationships[
            [
                "parcel_number",
                "entity_id",
                "role"
            ]
        ],
        on=[
            "parcel_number",
            "entity_id",
            "role"
        ],
        how="inner"
    )
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address"
            ]
        ],
        on="entity_id",
        how="left"
    )
    .sort_values(
        by=[
            "parcel_number",
            "entity_id",
            "role",
            "occurrence_id"
        ]
    )
)

duplicate_role_detail[
    [
        "parcel_number",
        "entity_id",
        "canonical_name",
        "entity_type",
        "role",
        "entity_name",
        "source_entity_name",
        "source_column",
        "associated_address",
        "joint_expanded",
        "source_occurrence_id",
        "occurrence_id"
    ]
]

In [ ]:
print(
    "Final entities:",
    len(entities)
)

print(
    "Entities represented in parcel roles:",
    parcel_entity_roles[
        "entity_id"
    ].nunique()
)

## Auditing merge_on_match_key

In [ ]:
raw_entities_audit = (
    resolver._create_raw_entities(
        entity_occurrences
    )
)

print(
    "Raw entities:",
    len(raw_entities_audit)
)

In [ ]:
pre_abbreviation_entities = (
    raw_entities_audit.copy()
)

pass_number = 0

while True:

    pass_number += 1

    count_before = len(
        pre_abbreviation_entities
    )

    pre_abbreviation_entities = (
        resolver._merge_blank_addresses(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_missing_street_suffix_variants(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_zip4_variants(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_equivalent_person_name_formats(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_middle_initial_variants(
            pre_abbreviation_entities
        )
    )

    count_after = len(
        pre_abbreviation_entities
    )

    print(
        f"Pass {pass_number}: "
        f"{count_before} -> {count_after}"
    )

    if count_after == count_before:
        break

In [ ]:
abbreviation_audit = (
    pre_abbreviation_entities.copy()
)

abbreviation_audit[
    "match_name"
] = (
    abbreviation_audit[
        "canonical_name"
    ]
    .apply(
        resolver._expand_common_abbreviations
    )
)

In [ ]:
abbreviation_candidates = []

for _, group in abbreviation_audit.groupby(
    [
        "entity_type",
        "match_name",
        "canonical_address"
    ],
    dropna=False,
    sort=False
):

    if len(group) < 2:
        continue

    original_names = tuple(
        sorted(
            set(
                group["canonical_name"]
            )
        )
    )

    # Ignore exact duplicates. We only want groups where
    # abbreviation expansion reconciles different names.
    if len(original_names) < 2:
        continue

    abbreviation_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "match_name":
            group["match_name"].iloc[0],

        "canonical_address":
            group["canonical_address"].iloc[0],

        "canonical_names":
            original_names,

        "entity_count":
            len(group),

        "occurrence_count":
            group[
                "occurrence_count"
            ].sum(),

        "row_indexes":
            tuple(
                group.index
            )
    })


abbreviation_candidates = pd.DataFrame(
    abbreviation_candidates
)

print(
    "Common-abbreviation candidate groups:",
    len(abbreviation_candidates)
)

abbreviation_candidates

In [ ]:
if not abbreviation_candidates.empty:

    print(
        abbreviation_candidates[
            "entity_type"
        ].value_counts()
    )

## Entities with the highest occurrence_count

In [ ]:
entities.sort_values(
    "occurrence_count",
    ascending=False
).head(10)

## Entities with canonical names containing a middle name or middle initial

In [ ]:
SUFFIXES = {
    "JR",
    "SR",
    "JUNIOR",
    "SENIOR",
    "II",
    "III",
    "IV",
}


def get_middle_tokens(name):
    """
    Return possible middle-name tokens from a person name.

    Supports both:
        SMITH, JOHN A
        SMITH JOHN A

    This is for manual inspection only, not entity-resolution logic.
    """

    if pd.isna(name):
        return tuple()

    name = str(name).upper().strip()
    name = name.replace(".", "")

    # Skip obvious joint/compound records
    if (
        "&" in name
        or "/" in name
        or re.search(r"\bAND\b", name)
    ):
        return tuple()

    if "," in name:

        _, given_part = name.split(",", 1)
        tokens = given_part.strip().split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # FIRST [MIDDLE...]
        return tuple(tokens[1:])

    else:

        tokens = name.split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # Assume LAST FIRST [MIDDLE...]
        if len(tokens) < 3:
            return tuple()

        return tuple(tokens[2:])

people_with_middle_names = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

people_with_middle_names["middle_tokens"] = (
    people_with_middle_names[
        "canonical_name"
    ]
    .apply(get_middle_tokens)
)

people_with_middle_names = (
    people_with_middle_names[
        people_with_middle_names[
            "middle_tokens"
        ].apply(len) > 0
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

people_with_middle_names[
    [
        "entity_id",
        "canonical_name",
        "middle_tokens",
        "canonical_address",
        "occurrence_count"
    ]
]

- Some names have commas while others don't
    - For example, KAVANAUGH, JAMES J vs. CLAWSON DORRINDA L
        - Note that both had their middle initial listed correctly under middle_tokens
- The canonical name FRITH, TERRANCE JEROME A produced the middle_token (JEROME, A)
- The canonical name BROWN, ROBERT WENDELL, JR. produced the middle_token (WENDELL,,)
    - Note that there are two commas rather than the usual one
- GREATER TRUE WORSHIP COGIC produced (WORSHIP, COGIC)
- BLACKSTAR STABILITY REVITALIZATION produced (REVITALIZATION,)
- EDWARDS, KARRIE; EDWARDS, KENTON produced (EDWARDS,, KENTON)

Unrelated case:
There are two entities with the name WIDGER JAMES LEROY, one with the address 214 MARSHALL ST, ALLEGAN, MI 49010 and the other with the address 214 MARSHALL ST, ALLEGAN, MI 49010. Because they have the exact same name and their address, with the removal of ST from the first case, are the same - they should clearly be merged.

### Distinguishing middle initials from full middle names

In [ ]:
def middle_type(tokens):

    if not tokens:
        return None

    if all(len(token) == 1 for token in tokens):
        return "Initial"

    return "Full/Other"


people_with_middle_names["middle_type"] = (
    people_with_middle_names[
        "middle_tokens"
    ]
    .apply(middle_type)
)

people_with_middle_names[
    "middle_type"
].value_counts()

Out of 3,955 unique entity names, 1,240 show as containing a middle name or initial.

In [ ]:
797 + 443

## Entities the previously had a blank address

In [ ]:
blank_address_cases = (
    resolution_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ],
        dropna=False
    )
    .agg(
        had_blank_address=(
            "associated_address",
            lambda x: (
                x.fillna("")
                .str.strip()
                .eq("")
                .any()
            )
        ),
        source_addresses=(
            "associated_address",
            lambda x: tuple(
                sorted(
                    {
                        address
                        for address in x.fillna("")
                        if address != ""
                    }
                )
            )
        ),
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        occurrence_count=(
            "occurrence_id",
            "size"
        )
    )
    .reset_index()
)

blank_address_cases = (
    blank_address_cases[
        blank_address_cases["had_blank_address"]
        &
        (
            blank_address_cases[
                "canonical_address"
            ] != ""
        )
    ]
)

print(blank_address_cases.shape)
blank_address_cases.head(50)

There are 1,580 cases of entities that used to have a blank address
- **QUESTION: Is the main thing to look for here that the source_names and canonical_name are in fact similar enough that we would want to merge them?**

## Names affected by CORP/CORPORATION, INC/INCORPORATED, CO/COMPANY

In [ ]:
ENTITY_NAME_REPLACEMENTS = {
    "CORP": "CORPORATION",
    "INC": "INCORPORATED",
    "CO": "COMPANY"
}


def expand_entity_abbreviations(name):

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    name = name.replace(".", "")

    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"\s*,\s*", ", ", name)

    for old, new in ENTITY_NAME_REPLACEMENTS.items():

        name = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            name
        )

    return name.strip()

abbreviation_audit = resolution_audit.copy()

abbreviation_audit["expanded_name"] = (
    abbreviation_audit[
        "entity_name"
    ]
    .apply(expand_entity_abbreviations)
)

abbreviation_cases = (
    abbreviation_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        expanded_names=(
            "expanded_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        source_name_count=(
            "entity_name",
            "nunique"
        ),
        expanded_name_count=(
            "expanded_name",
            "nunique"
        )
    )
    .reset_index()
)

abbreviation_cases = (
    abbreviation_cases[
        (abbreviation_cases["source_name_count"] > 1)
        &
        (abbreviation_cases["expanded_name_count"] == 1)
    ]
)

abbreviation_cases.head(50)

- There is 1 case of a CORP/CORPORATION merger.
- Other than that, the other 5 cases involve merging source names that differ by the inclusion/exclusion of a period.
    - For example, ONE DAY, LLC and ONE DAY, LLC.

## Suffix cases

### All people containing a suffix

In [ ]:
suffix_pattern = (
    r"\b(?:JR|SR|JUNIOR|SENIOR|II|III|IV)\b"
)

suffix_entities = (
    entities[
        (entities["entity_type"] == "Person")
        &
        (
            entities["canonical_name"]
            .str.contains(
                suffix_pattern,
                regex=True,
                na=False
            )
        )
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

print(suffix_entities.shape)

suffix_entities[
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
].head(50)

- 146 entities contain suffixes

### Entities with both suffix and suffix-free versions

In [ ]:
suffix_audit = (
    resolution_audit[
        resolution_audit[
            "entity_type_final"
        ] == "Person"
    ]
    .copy()
)

suffix_audit["has_suffix"] = (
    suffix_audit[
        "entity_name"
    ]
    .str.contains(
        suffix_pattern,
        regex=True,
        na=False
    )
)

suffix_merge_cases = (
    suffix_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        has_suffix_version=(
            "has_suffix",
            "any"
        ),
        has_no_suffix_version=(
            "has_suffix",
            lambda x: (~x).any()
        )
    )
    .reset_index()
)

suffix_merge_cases = (
    suffix_merge_cases[
        suffix_merge_cases["has_suffix_version"]
        &
        suffix_merge_cases["has_no_suffix_version"]
    ]
)

suffix_merge_cases.head(50)

- There is 1 case of an entity merged due to having the same name and address, with different suffixes
    - Albert Derring and Albert Derring III

## Cases of finalized entities with the same name but different associated addresses

In [ ]:
person_entities = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

known_address_people = (
    person_entities[
        person_entities[
            "canonical_address"
        ].fillna("").str.strip() != ""
    ]
)

common_names_multiple_addresses = (
    known_address_people
    .groupby("canonical_name")
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),
        address_count=(
            "canonical_address",
            "nunique"
        ),
        addresses=(
            "canonical_address",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        entity_ids=(
            "entity_id",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        total_occurrences=(
            "occurrence_count",
            "sum"
        )
    )
    .reset_index()
)

common_names_multiple_addresses = (
    common_names_multiple_addresses[
        common_names_multiple_addresses[
            "address_count"
        ] > 1
    ]
    .sort_values(
        [
            "address_count",
            "total_occurrences"
        ],
        ascending=False
    )
)

print(common_names_multiple_addresses.shape)
common_names_multiple_addresses.head(50)

- There are 296 canonical_names which are associated with more than one entity/address

## Inspecting name formatting

In [ ]:
people = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Person"
    ]
)

comma_names = (
    people["entity_name"]
    .str.contains(",", regex=False, na=False)
)

print(
    f"Person occurrences with comma-format names: "
    f"{comma_names.sum():,}"
)

print(
    f"Person occurrences without comma-format names: "
    f"{(~comma_names).sum():,}"
)

print(
    f"Percent parseable by current comma requirement: "
    f"{comma_names.mean():.1%}"
)

## Inspecting unknown entity types

In [ ]:
entity_occurrences[
    "entity_type"
].value_counts()

In [ ]:
unknown_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Unknown",
        "entity_name"
    ]
    .value_counts()
    .rename_axis("entity_name")
    .reset_index(name="occurrence_count")
)

print(
    f"Unique Unknown names: "
    f"{len(unknown_names):,}"
)

unknown_names.head(100)

In [ ]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences[
            "entity_type"
        ] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
)

joint_names.sample(
    min(100, len(joint_names)),
    random_state=42
).sort_values()

In [ ]:
joint_summary = pd.Series({
    "Contains &": joint_names.str.contains(
        "&",
        regex=False
    ).sum(),

    "Contains AND": joint_names.str.contains(
        r"\bAND\b",
        regex=True
    ).sum(),

    "Contains /": joint_names.str.contains(
        "/",
        regex=False
    ).sum(),

    "Contains comma": joint_names.str.contains(
        ",",
        regex=False
    ).sum(),
})

joint_summary

## Inspecting Joint type entities

In [ ]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

def profile_joint_name(name):

    candidate = resolver._normalize_entity_type_candidate(name)

    return pd.Series({
        "ampersand_count": candidate.count("&"),
        "semicolon_count": candidate.count(";"),
        "slash_count": candidate.count("/"),

        "and_count": len(
            re.findall(
                r"\bAND\b",
                candidate
            )
        ),

        "comma_count": candidate.count(","),

        "contains_et_al": bool(
            re.search(
                r"\bET AL\b",
                candidate
            )
        ),

        "token_count": len(
            candidate.split()
        )
    })

joint_profile = (
    joint_names
    .to_frame(name="entity_name")
)

joint_profile = pd.concat(
    [
        joint_profile,

        joint_profile["entity_name"]
        .apply(profile_joint_name)
    ],
    axis=1
)

joint_profile.head()

In [ ]:
joint_profile["separator_type"] = "Other"

joint_profile.loc[
    (
        joint_profile["ampersand_count"] == 1
    )
    &
    (
        joint_profile[
            [
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) == 0
    ),
    "separator_type"
] = "Single ampersand"

joint_profile.loc[
    joint_profile["semicolon_count"] > 0,
    "separator_type"
] = "Semicolon"

joint_profile.loc[
    joint_profile["slash_count"] > 0,
    "separator_type"
] = "Slash"

joint_profile.loc[
    joint_profile["and_count"] > 0,
    "separator_type"
] = "AND"

joint_profile.loc[
    (
        joint_profile[
            [
                "ampersand_count",
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) > 1
    ),
    "separator_type"
] = "Multiple separators"

joint_profile["separator_type"].value_counts()

In [ ]:
simple_ampersand = (
    joint_profile[
        (
            joint_profile["separator_type"]
            == "Single ampersand"
        )
        &
        (~joint_profile["contains_et_al"])
    ]
)

print(
    f"Unique simple ampersand Joint names: "
    f"{len(simple_ampersand):,}"
)

simple_ampersand[
    [
        "entity_name",
        "comma_count",
        "token_count"
    ]
].sample(
    min(100, len(simple_ampersand)),
    random_state=42
)

In [ ]:
simple_ampersand["comma_count"].value_counts().sort_index()

In [ ]:
simple_ampersand.groupby(
    "comma_count"
)["token_count"].describe()

In [ ]:
print(len(simple_ampersand))

print(
    simple_ampersand["comma_count"]
    .isna()
    .sum()
)

In [ ]:
medium_joint_candidates = (
    entity_occurrences[
        (entity_occurrences["entity_type"] == "Joint")
        &
        (
            entity_occurrences["joint_parse_confidence"]
            == "Medium"
        )
    ]
    .copy()
)

print(
    "Medium-confidence Joint occurrences:",
    f"{len(medium_joint_candidates):,}"
)

print(
    "Unique Medium-confidence Joint names:",
    f"{medium_joint_candidates['entity_name'].nunique():,}"
)

In [ ]:
medium_joint_candidates[
    "joint_parse_rule"
].value_counts()

In [ ]:
medium_joint_names = (
    medium_joint_candidates[
        [
            "entity_name",
            "joint_parse_rule"
        ]
    ]
    .drop_duplicates()
)

medium_joint_names.sample(
    min(100, len(medium_joint_names)),
    random_state=42
).sort_values(
    "entity_name"
)

In [ ]:
expanded_joint_audit = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    [
        [
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "joint_party_index",
            "joint_party_count",
            "joint_parse_rule",
            "joint_parse_confidence"
        ]
    ]
    .sort_values(
        [
            "source_occurrence_id",
            "joint_party_index"
        ]
    )
)

expanded_joint_audit.head(100)

In [ ]:
expanded_joint_pairs = (
    expanded_joint_audit
    .pivot_table(
        index=[
            "source_occurrence_id",
            "source_entity_name",
            "joint_parse_rule",
            "joint_parse_confidence"
        ],
        columns="joint_party_index",
        values="entity_name",
        aggfunc="first"
    )
    .reset_index()
    .rename(
        columns={
            1: "person_1",
            2: "person_2"
        }
    )
)

medium_joint_names.sort_values(
    "entity_name"
).reset_index(
    drop=True
)

In [ ]:
expanded_counts = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    .groupby(
        "source_occurrence_id"
    )
    .size()
)

print(
    "Expanded source occurrences not producing exactly 2 rows:",
    (expanded_counts != 2).sum()
)

In [ ]:
expanded_source_ids = set(
    entity_occurrences.loc[
        entity_occurrences["joint_expanded"],
        "source_occurrence_id"
    ]
)

surviving_joint_rows = (
    entity_occurrences[
        entity_occurrences["source_occurrence_id"]
        .isin(expanded_source_ids)
        &
        (
            entity_occurrences["entity_type"]
            == "Joint"
        )
    ]
)

print(
    "Expanded sources still containing a Joint row:",
    len(surviving_joint_rows)
)

In [ ]:
entity_occurrences["entity_type"].value_counts()

## Analyzing Joint type entities after splitting high and medium confidence one and two comma Joint entities

In [ ]:
remaining_joint = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Joint"
    ]
    .copy()
)

print(
    "Remaining Joint occurrences:",
    f"{len(remaining_joint):,}"
)

print(
    "Unique remaining Joint names:",
    f"{remaining_joint['entity_name'].nunique():,}"
)

In [ ]:
remaining_joint_names = (
    remaining_joint["entity_name"]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
remaining_parseable = (
    remaining_joint_names
    .apply(
        lambda name:
        resolver._parse_joint_name(name)
        is not None
    )
)

print(
    "Remaining Joint names still parseable "
    "by current parser:",
    remaining_parseable.sum()
)

In [ ]:
def profile_joint_name(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    return pd.Series({
        "ampersand_count":
            candidate.count("&"),

        "semicolon_count":
            candidate.count(";"),

        "slash_count":
            candidate.count("/"),

        "and_count":
            len(
                re.findall(
                    r"\bAND\b",
                    candidate
                )
            ),

        "comma_count":
            candidate.count(","),

        "contains_et_al":
            bool(
                re.search(
                    r"\bET AL\b",
                    candidate
                )
            ),

        "token_count":
            len(candidate.split())
    })

In [ ]:
remaining_joint_profile = (
    remaining_joint_names
    .to_frame(name="entity_name")
)

remaining_joint_profile = pd.concat(
    [
        remaining_joint_profile,

        remaining_joint_profile[
            "entity_name"
        ].apply(
            profile_joint_name
        )
    ],
    axis=1
)

In [ ]:
remaining_joint_profile[
    "comma_count"
].value_counts().sort_index()

In [ ]:
remaining_joint_profile[
    [
        "ampersand_count",
        "semicolon_count",
        "slash_count",
        "and_count"
    ]
].describe()

In [ ]:
zero_comma_single_ampersand = (
    remaining_joint_profile[
        (remaining_joint_profile["ampersand_count"] == 1)
        &
        (remaining_joint_profile["comma_count"] == 0)
        &
        (remaining_joint_profile["semicolon_count"] == 0)
        &
        (remaining_joint_profile["slash_count"] == 0)
        &
        (remaining_joint_profile["and_count"] == 0)
        &
        (~remaining_joint_profile["contains_et_al"])
    ]
    .copy()
)

print(
    "Unique zero-comma single-ampersand names:",
    f"{len(zero_comma_single_ampersand):,}"
)

In [ ]:
zero_comma_single_ampersand[
    [
        "entity_name",
        "token_count"
    ]
].sample(
    min(
        100,
        len(zero_comma_single_ampersand)
    ),
    random_state=42
).sort_values(
    "entity_name"
)

## Analyzing single ampersand cases with no commas

In [ ]:
def profile_zero_comma_joint(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    left, right = [
        part.strip()
        for part in candidate.split("&", 1)
    ]

    left_tokens = left.split()
    right_tokens = right.split()

    return pd.Series({
        "left": left,
        "right": right,

        "left_token_count":
            len(left_tokens),

        "right_token_count":
            len(right_tokens),

        "left_empty":
            len(left_tokens) == 0,

        "right_empty":
            len(right_tokens) == 0,

        "left_first_token":
            left_tokens[0]
            if left_tokens
            else "",

        "right_first_token":
            right_tokens[0]
            if right_tokens
            else "",

        "right_last_token":
            right_tokens[-1]
            if right_tokens
            else "",

        "right_ends_initial":
            bool(
                right_tokens
                and len(right_tokens[-1]) == 1
            ),

        "same_leading_token":
            bool(
                left_tokens
                and right_tokens
                and left_tokens[0]
                == right_tokens[0]
            )
    })


zero_comma_structure = pd.concat(
    [
        zero_comma_single_ampersand[
            ["entity_name"]
        ].reset_index(drop=True),

        zero_comma_single_ampersand[
            "entity_name"
        ]
        .reset_index(drop=True)
        .apply(profile_zero_comma_joint)
    ],
    axis=1
)

In [ ]:
zero_comma_structure[
    "right_token_count"
].value_counts().sort_index()

In [ ]:
pd.crosstab(
    zero_comma_structure[
        "right_token_count"
    ],
    zero_comma_structure[
        "right_ends_initial"
    ]
)

In [ ]:
print(
    "Trailing-empty records:",
    zero_comma_structure[
        "right_empty"
    ].sum()
)

print(
    "Same leading token on both sides:",
    zero_comma_structure[
        "same_leading_token"
    ].sum()
)

### Continuing analysis

In [ ]:
pd.crosstab(
    zero_comma_structure["left_token_count"],
    zero_comma_structure["right_token_count"]
)

left_one_token = (
    zero_comma_structure[
        (zero_comma_structure["left_token_count"] == 1)
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

print(
    "Unique names with one token on the left:",
    len(left_one_token)
)

left_one_token[
    [
        "entity_name",
        "left",
        "right",
        "right_token_count"
    ]
].sort_values(
    "entity_name"
)

In [ ]:
right_two_ending_initial = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (zero_comma_structure["right_ends_initial"])
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

right_two_ending_initial[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

In [ ]:
right_two_full_tokens = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (~zero_comma_structure["right_ends_initial"])
    ]
    .copy()
)

right_two_full_tokens[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

In [ ]:
entity_occurrences.loc[
    entity_occurrences["joint_expanded"],
    "joint_parse_rule"
].value_counts()

## Investigating building a name dictionary from county data

In [ ]:
known_first_names = set()
known_last_names = set()

for name in (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Person",
        "entity_name"
    ]
    .drop_duplicates()
):

    normalized = resolver._normalize_match_name(
        name
    )

    # Only use comma-format names for building the vocabulary.
    # Their name orientation is explicitly represented in the
    # source rather than inferred.
    if "," not in normalized:
        continue

    parsed = resolver._parse_person_name(
        normalized
    )

    if parsed is None:
        continue

    known_first_names.add(
        parsed["first_name"]
    )

    known_last_names.add(
        parsed["last_name"]
    )

In [ ]:
def get_name_position_evidence(row):

    right_tokens = row["right"].split()

    if len(right_tokens) != 2:
        return pd.Series({
            "right_token_1": "",
            "right_token_2": "",
            "token_1_seen_first": False,
            "token_1_seen_last": False,
            "token_2_seen_first": False,
            "token_2_seen_last": False
        })

    token_1, token_2 = right_tokens

    return pd.Series({
        "right_token_1": token_1,
        "right_token_2": token_2,

        "token_1_seen_first":
            token_1 in known_first_names,

        "token_1_seen_last":
            token_1 in known_last_names,

        "token_2_seen_first":
            token_2 in known_first_names,

        "token_2_seen_last":
            token_2 in known_last_names
    })


right_two_full_evidence = pd.concat(
    [
        right_two_full_tokens.reset_index(
            drop=True
        ),

        right_two_full_tokens.reset_index(
            drop=True
        ).apply(
            get_name_position_evidence,
            axis=1
        )
    ],
    axis=1
)

In [ ]:
def classify_right_side_evidence(row):

    # Strong evidence for:
    #
    # SURNAME FIRST
    #
    # e.g. MCCADEN SHAWNA
    if (
        row["token_1_seen_last"]
        and row["token_2_seen_first"]
        and not row["token_1_seen_first"]
    ):
        return "Likely separate surname"

    # Strong evidence for:
    #
    # FIRST MIDDLE
    #
    # e.g. MARY ANNE
    if (
        row["token_1_seen_first"]
        and not row["token_1_seen_last"]
        and not row["token_2_seen_last"]
    ):
        return "Likely shared surname"

    return "Ambiguous"


right_two_full_evidence[
    "name_evidence"
] = right_two_full_evidence.apply(
    classify_right_side_evidence,
    axis=1
)

In [ ]:
right_two_full_evidence[
    "name_evidence"
].value_counts()

In [ ]:
right_two_full_evidence[
    [
        "entity_name",
        "right",
        "right_token_1",
        "right_token_2",
        "token_1_seen_first",
        "token_1_seen_last",
        "token_2_seen_first",
        "token_2_seen_last",
        "name_evidence"
    ]
].sort_values(
    [
        "name_evidence",
        "entity_name"
    ]
)

### Adding address data

In [ ]:
person_evidence_rows = []

for _, row in entity_occurrences.iterrows():

    # Avoid using Joint-derived people to prove our own Joint
    # decomposition assumptions.
    if row["entity_type"] != "Person":
        continue

    if row.get("joint_expanded", False):
        continue

    parsed = resolver._parse_person_name(
        row["entity_name"]
    )

    if parsed is None:
        continue

    person_evidence_rows.append({
        "entity_name": row["entity_name"],
        "associated_address": row["associated_address"],
        "last_name": parsed["last_name"],
        "first_name": parsed["first_name"],
        "middle_tokens": parsed["middle_tokens"],
        "suffix": parsed["suffix"]
    })


person_evidence = pd.DataFrame(
    person_evidence_rows
)

In [ ]:
def evaluate_two_full_token_joint(row):

    left_tokens = row["left"].split()
    right_tokens = row["right"].split()

    result = {
        "shared_candidate": "",
        "separate_candidate": "",
        "shared_exact_anywhere": False,
        "separate_exact_anywhere": False,
        "shared_same_address": False,
        "separate_same_address": False
    }

    if (
        len(left_tokens) < 2
        or len(right_tokens) != 2
    ):
        return pd.Series(result)

    shared_surname = left_tokens[0]

    right_1 = right_tokens[0]
    right_2 = right_tokens[1]

    # Hypothesis A:
    # right side is FIRST MIDDLE and inherits left surname.
    shared_candidate = (
        f"{shared_surname}, "
        f"{right_1} {right_2}"
    )

    # Hypothesis B:
    # right side is LAST FIRST.
    separate_candidate = (
        f"{right_1}, "
        f"{right_2}"
    )

    result[
        "shared_candidate"
    ] = shared_candidate

    result[
        "separate_candidate"
    ] = separate_candidate

    shared_parsed = (
        resolver._parse_person_name(
            shared_candidate
        )
    )

    separate_parsed = (
        resolver._parse_person_name(
            separate_candidate
        )
    )

    address = row.get(
        "associated_address",
        ""
    )

    if shared_parsed is not None:

        shared_matches = person_evidence[
            (person_evidence["last_name"]
             == shared_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == shared_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == shared_parsed["middle_tokens"])
        ]

        result[
            "shared_exact_anywhere"
        ] = len(shared_matches) > 0

        result[
            "shared_same_address"
        ] = (
            (
                shared_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    if separate_parsed is not None:

        separate_matches = person_evidence[
            (person_evidence["last_name"]
             == separate_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == separate_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == separate_parsed["middle_tokens"])
        ]

        result[
            "separate_exact_anywhere"
        ] = len(separate_matches) > 0

        result[
            "separate_same_address"
        ] = (
            (
                separate_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    return pd.Series(result)

In [ ]:
right_two_full_with_address = (
    right_two_full_tokens
    .merge(
        remaining_joint[
            [
                "entity_name",
                "associated_address"
            ]
        ].drop_duplicates(),
        on="entity_name",
        how="left"
    )
)

right_two_full_with_address

In [ ]:
context_evidence = pd.concat(
    [
        right_two_full_with_address
        .reset_index(drop=True),

        right_two_full_with_address
        .reset_index(drop=True)
        .apply(
            evaluate_two_full_token_joint,
            axis=1
        )
    ],
    axis=1
)

context_evidence

In [ ]:
context_evidence_by_name = (
    context_evidence
    .groupby(
        "entity_name",
        as_index=False
    )
    .agg(
        shared_candidate=(
            "shared_candidate",
            "first"
        ),
        separate_candidate=(
            "separate_candidate",
            "first"
        ),

        shared_exact_anywhere=(
            "shared_exact_anywhere",
            "any"
        ),
        separate_exact_anywhere=(
            "separate_exact_anywhere",
            "any"
        ),

        shared_same_address=(
            "shared_same_address",
            "any"
        ),
        separate_same_address=(
            "separate_same_address",
            "any"
        ),

        address_count=(
            "associated_address",
            "nunique"
        )
    )
)

In [ ]:
print(
    "Unique Joint names in contextual analysis:",
    len(context_evidence_by_name)
)

In [ ]:
pd.crosstab(
    context_evidence_by_name[
        "shared_same_address"
    ],
    context_evidence_by_name[
        "separate_same_address"
    ],
    margins=True
)

In [ ]:
same_address_evidence = (
    context_evidence_by_name[
        context_evidence_by_name[
            [
                "shared_same_address",
                "separate_same_address"
            ]
        ].any(axis=1)
    ]
    [
        [
            "entity_name",
            "shared_candidate",
            "separate_candidate",
            "shared_same_address",
            "separate_same_address",
            "shared_exact_anywhere",
            "separate_exact_anywhere",
            "address_count"
        ]
    ]
    .sort_values(
        [
            "shared_same_address",
            "separate_same_address",
            "entity_name"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
)

same_address_evidence

In [ ]:
def classify_contextual_evidence(row):

    shared_same = row[
        "shared_same_address"
    ]

    separate_same = row[
        "separate_same_address"
    ]

    shared_any = row[
        "shared_exact_anywhere"
    ]

    separate_any = row[
        "separate_exact_anywhere"
    ]


    # ------------------------------------------------------
    # Strongest evidence: same address
    # ------------------------------------------------------

    if shared_same and not separate_same:
        return "Shared surname - same address"

    if separate_same and not shared_same:
        return "Separate surnames - same address"

    if shared_same and separate_same:
        return "Conflict - both same address"


    # ------------------------------------------------------
    # Secondary evidence: exact candidate elsewhere
    # ------------------------------------------------------

    if shared_any and not separate_any:
        return "Shared surname - elsewhere"

    if separate_any and not shared_any:
        return "Separate surnames - elsewhere"

    if shared_any and separate_any:
        return "Conflict - both elsewhere"


    # ------------------------------------------------------
    # No contextual evidence
    # ------------------------------------------------------

    return "No contextual evidence"

In [ ]:
context_evidence_by_name[
    "context_decision"
] = (
    context_evidence_by_name
    .apply(
        classify_contextual_evidence,
        axis=1
    )
)

In [ ]:
context_evidence_by_name[
    "context_decision"
].value_counts()

## Addressing cases of same name similar addresses

In [ ]:
STREET_SUFFIXES = {
    "ST",
    "AVE",
    "RD",
    "DR",
    "BLVD",
    "LN",
    "CT",
    "CIR",
    "PL",
    "PKWY",
    "TER",
    "HWY",
    "WAY",
    "TRL"
}

UNIT_MARKERS = {
    "APT",
    "STE",
    "UNIT",
    "BLDG",
    "FL",
    "RM"
}


def parse_address_suffix_for_audit(address):
    """
    Break a cleaned address into components needed to detect
    cases where a street suffix is present in one representation
    and absent in another.

    This is for analysis only and does not modify the address.
    """

    empty_result = pd.Series({
        "street_base": None,
        "street_suffix": None,
        "unit": None,
        "locality": None
    })

    if pd.isna(address):
        return empty_result

    address = str(address).strip()

    if not address:
        return empty_result

    parts = [
        part.strip()
        for part in address.split(",")
    ]

    # Require at least:
    #
    # street, locality...
    if len(parts) < 2:
        return empty_result

    street_line = parts[0]

    # Don't try to interpret PO Boxes as street addresses.
    if street_line.startswith("PO BOX"):
        return empty_result

    locality = ", ".join(parts[1:])

    tokens = street_line.split()

    if len(tokens) < 2:
        return empty_result


    # ----------------------------------------------------------
    # Separate unit information
    # ----------------------------------------------------------

    unit_index = None

    for i, token in enumerate(tokens):

        if (
            token in UNIT_MARKERS
            or token.startswith("#")
        ):
            unit_index = i
            break

    if unit_index is not None:

        street_tokens = tokens[:unit_index]
        unit_tokens = tokens[unit_index:]

    else:

        street_tokens = tokens
        unit_tokens = []


    if len(street_tokens) < 2:
        return empty_result


    # Require something resembling a street number.
    if not any(
        character.isdigit()
        for character in street_tokens[0]
    ):
        return empty_result


    # ----------------------------------------------------------
    # Identify street suffix
    # ----------------------------------------------------------

    if street_tokens[-1] in STREET_SUFFIXES:

        street_suffix = street_tokens[-1]

        street_base_tokens = (
            street_tokens[:-1]
        )

    else:

        street_suffix = ""

        street_base_tokens = street_tokens


    if not street_base_tokens:
        return empty_result


    return pd.Series({
        "street_base":
            " ".join(street_base_tokens),

        "street_suffix":
            street_suffix,

        "unit":
            " ".join(unit_tokens),

        "locality":
            locality
    })

In [ ]:
address_suffix_audit = (
    entities[
        entities["canonical_address"]
        .fillna("")
        .str.strip()
        .ne("")
    ]
    .copy()
)

parsed_addresses = (
    address_suffix_audit[
        "canonical_address"
    ]
    .apply(
        parse_address_suffix_for_audit
    )
)

address_suffix_audit = pd.concat(
    [
        address_suffix_audit.reset_index(
            drop=True
        ),

        parsed_addresses.reset_index(
            drop=True
        )
    ],
    axis=1
)

In [ ]:
address_suffix_candidates = []

group_columns = [
    "entity_type",
    "canonical_name",
    "street_base",
    "unit",
    "locality"
]

for _, group in (
    address_suffix_audit
    .dropna(
        subset=[
            "street_base",
            "street_suffix",
            "locality"
        ]
    )
    .groupby(
        group_columns,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue

    suffixes = set(
        group["street_suffix"]
    )

    nonblank_suffixes = {
        suffix
        for suffix in suffixes
        if suffix
    }

    # Require:
    #
    #   one suffix-free version
    #   +
    #   exactly one known suffix
    #
    # Example:
    #
    #   214 MARSHALL
    #   214 MARSHALL ST
    #
    if (
        "" not in suffixes
        or len(nonblank_suffixes) != 1
    ):
        continue

    address_suffix_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "canonical_name":
            group["canonical_name"].iloc[0],

        "street_base":
            group["street_base"].iloc[0],

        "locality":
            group["locality"].iloc[0],

        "unit":
            group["unit"].iloc[0],

        "known_suffix":
            next(iter(nonblank_suffixes)),

        "entity_ids":
            tuple(
                group["entity_id"]
            ),

        "addresses":
            tuple(
                group["canonical_address"]
            ),

        "entity_count":
            len(group)
    })


address_suffix_candidates = pd.DataFrame(
    address_suffix_candidates
)

In [ ]:
print(
    "Candidate missing-street-suffix groups:",
    len(address_suffix_candidates)
)

address_suffix_candidates

In [ ]:
entities["occurrence_count"].sum()

In [ ]:
entities[
    entities["canonical_name"]
    .str.contains(
        "WIDGER JAMES LEROY",
        regex=False,
        na=False
    )
]

In [ ]:
entities.head()
entities.info()

In [ ]:
print(
    "Unique source occurrences:",
    entity_occurrences["source_occurrence_id"].nunique()
)

print(
    "Entity occurrences after Joint expansion:",
    len(entity_occurrences)
)

print(
    "Unique occurrence IDs:",
    entity_occurrences["occurrence_id"].nunique()
)

print(
    "Sum of final entity occurrence counts:",
    entities["occurrence_count"].sum()
)

## Suffix merge change analysis

In [ ]:
entities[
    entities["canonical_name"]
    .str.contains(
        "DERRING",
        regex=False,
        na=False
    )
][
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
]

In [ ]:
entities[
    entities["canonical_name"]
    .str.contains(
        "LINDSEY, WILLIAM B",
        regex=False,
        na=False
    )
][
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
]

In [ ]:
entities["occurrence_count"].sum()
# 12058

## Address zip code auditing

In [ ]:
import re


def parse_zip_variant_for_audit(address):
    """
    Extract the address body and ZIP components for auditing
    5-digit ZIP versus ZIP+4 variants.

    Does not modify the original address.
    """

    if pd.isna(address):
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    address = str(address).strip()

    if not address:
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    match = re.search(
        r"^(.*\D)(\d{5})(?:-(\d{4}))?$",
        address
    )

    if match is None:
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    return pd.Series({
        "address_without_zip":
            match.group(1).strip(),

        "zip5":
            match.group(2),

        "zip4":
            match.group(3) or ""
    })

In [ ]:
zip_audit = (
    entities[
        entities["canonical_address"]
        .fillna("")
        .str.strip()
        .ne("")
    ]
    .copy()
    .reset_index(drop=True)
)

zip_parts = (
    zip_audit["canonical_address"]
    .apply(parse_zip_variant_for_audit)
)

zip_audit = pd.concat(
    [
        zip_audit,
        zip_parts
    ],
    axis=1
)

In [ ]:
zip_variant_candidates = []

group_columns = [
    "entity_type",
    "canonical_name",
    "address_without_zip",
    "zip5"
]

for _, group in (
    zip_audit
    .dropna(
        subset=[
            "address_without_zip",
            "zip5"
        ]
    )
    .groupby(
        group_columns,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue

    zip4_values = set(
        group["zip4"]
    )

    nonblank_zip4 = {
        value
        for value in zip4_values
        if value
    }

    # Require:
    #
    #     49048
    #     49048-4827
    #
    # but reject a group containing conflicting ZIP+4 values.
    if (
        "" not in zip4_values
        or len(nonblank_zip4) != 1
    ):
        continue

    zip_variant_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "canonical_name":
            group["canonical_name"].iloc[0],

        "address_without_zip":
            group["address_without_zip"].iloc[0],

        "zip5":
            group["zip5"].iloc[0],

        "known_zip4":
            next(iter(nonblank_zip4)),

        "addresses":
            tuple(
                group["canonical_address"]
                .drop_duplicates()
            ),

        "entity_ids":
            tuple(
                group["entity_id"]
            ),

        "entity_count":
            len(group)
    })


zip_variant_candidates = pd.DataFrame(
    zip_variant_candidates
)

print(
    "Candidate ZIP+4 variant groups:",
    len(zip_variant_candidates)
)

zip_variant_candidates

## Testing collapse_index_groups change

In [ ]:
test_entities = pd.DataFrame({
    "canonical_name": [
        "SMITH, JOHN",
        "SMITH, JOHN A",
        "SMITH, JOHN ALAN"
    ],

    "entity_type": [
        "Person",
        "Person",
        "Person"
    ],

    "canonical_address": [
        "123 MAIN ST, KALAMAZOO, MI 49001",
        "123 MAIN ST, KALAMAZOO, MI 49001",
        "123 MAIN ST, KALAMAZOO, MI 49001"
    ],

    "organization_type": [
        None,
        None,
        None
    ],

    "organization_classification_confidence": [
        None,
        None,
        None
    ],

    "occurrence_count": [
        1,
        1,
        1
    ],

    "_occurrence_ids": [
        (1,),
        (2,),
        (3,)
    ]
})

In [ ]:
resolver._collapse_index_groups(
    test_entities,
    [
        [0, 1],
        [0, 2]
    ]
)

In [ ]:
resolver._collapse_index_groups(
    test_entities,
    [
        [0, 1],
        [2, 0]
    ]
)

In [ ]:
groups_to_merge = [
    [0, 1],
    [2, 3]
]

In [ ]:
entities["occurrence_count"].sum()
# 12058